# Erster Test: Gemini 3.8 Flash mit den sechs Testfragen

Ziel: grob prüfen, ob Gemini 3.8 Flash die geprüften Testfragen in `data/synthetic_reports/test_questions.md` sinnvoll beantwortet.

- **Noch keine Suche (Retrieval):** Alle fünf fiktiven Berichte gehen vollständig in den Prompt. So sieht man, was das Modell kann, bevor die Suche als zusätzliche Fehlerquelle dazukommt.
- **Daten:** Nur die fiktiven Testberichte aus diesem Repository. Im kostenlosen Kontingent kann Google die Eingaben zur Produktverbesserung nutzen.
- **Schlüssel:** Wird aus den Colab-Secrets gelesen (Schlüsselsymbol links). Den Namen in `KEY_NAME` anpassen, falls dein Secret anders heißt.
- **Bewertung:** Antworten danach von Hand mit den Antwortentwürfen und Kriterien in `test_questions.md` vergleichen.

In [ ]:
!pip install -q -U google-genai
!git clone -q https://github.com/Julien0049/AI-database.git || git -C AI-database pull -q

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

KEY_NAME = "GOOGLE_API_KEY"
MODEL = "gemini-3.8-flash"

client = genai.Client(api_key=userdata.get(KEY_NAME))

In [ ]:
import pathlib, re

DATA = pathlib.Path("AI-database/data/synthetic_reports")
reports = sorted(DATA.glob("AAR-*.md"))
corpus = "\n\n".join(
    f'<dokument datei="{p.name}">\n{p.read_text()}\n</dokument>' for p in reports
)

# Die Fragen stehen in test_questions.md jeweils als Zitat unter "**Frage**".
questions = re.findall(r"\*\*Frage\*\*\s*\n\s*> (.+)", (DATA / "test_questions.md").read_text())
print(len(reports), "Berichte,", len(questions), "Fragen")

## Antwortregeln

Der Systemprompt übernimmt die bestätigten Antwortregeln aus `AGENTS.md`.

In [ ]:
SYSTEM = """Du beantwortest Fragen ausschließlich auf Grundlage der mitgelieferten Berichte.
Die Berichte sind fiktive Testdaten.

Regeln:
1. Verwende nur Aussagen aus den Berichten. Ergänze nichts aus eigenem Wissen.
2. Ergänzende Aussagen aus mehreren Berichten darfst du zu einer klaren Antwort zusammenführen.
3. Wenn Berichte mit vergleichbarem Kontext sich widersprechen, bestimmt die jüngste passende Aussage die Antwort.
   Nenne ältere abweichende Verfahren nicht als gleichrangige Alternativen.
4. Wenn Berichte unterschiedliche Kontexte betreffen oder keine sichere Priorisierung zulassen, behaupte keine Priorität.
5. Bezeichne keine Quelle als gültig oder verbindlich; die Berichte belegen das nicht.
6. Wenn die Berichte die Frage nicht ausreichend beantworten, sag das offen und fülle keine Lücken.

Format: Zuerst die knappe Antwort. Danach eine Zeile "Quellen:" mit Dateiname und Datum der Berichte, die die Antwort stützen."""

In [ ]:
import time

results = []
for i, question in enumerate(questions, 1):
    response = client.models.generate_content(
        model=MODEL,
        contents=f"{corpus}\n\nFrage: {question}",
        config=types.GenerateContentConfig(system_instruction=SYSTEM),
    )
    results.append({"nr": i, "frage": question, "antwort": response.text})
    print(f"## Frage {i}\n{question}\n\n{response.text}\n")
    time.sleep(5)  # Abstand wegen der Limits im kostenlosen Kontingent

## Ergebnisse speichern

Lädt eine JSON-Datei mit Modell, Zeitpunkt, Systemprompt und Antworten herunter. Sie kann zur Auswertung ins Repository übernommen werden.

In [ ]:
import json, datetime
from google.colab import files

run = {
    "modell": MODEL,
    "zeitpunkt": datetime.datetime.now().isoformat(timespec="seconds"),
    "systemprompt": SYSTEM,
    "ergebnisse": results,
}
name = f"ergebnisse_{MODEL}_{datetime.date.today()}.json"
pathlib.Path(name).write_text(json.dumps(run, ensure_ascii=False, indent=2))
files.download(name)